# Variable-length audio records with TorchSIG

This example downloads two short, intelligible English speech recordings, plays them, converts each recording to stereo I/Q audio, describes variable-length segments in `metadata.csv`, and reads those segments with TorchSIG's `WAVReader`. It demonstrates the audio-record manifest introduced in MR2: records can have different lengths, multiple records can share a file, files can live in nested directories, and different files can contain different record counts.

The speech comes from [TorchAudio's tutorial assets](https://github.com/pytorch/audio/tree/main/examples/tutorials) and [OpenAI Whisper's JFK test clip](https://github.com/openai/whisper/blob/main/tests/jfk.flac). The source recordings are played unchanged. For the TorchSIG dataset, a Hilbert transform puts the original speech in I and its quadrature component in Q.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import csv

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from scipy.signal import hilbert
from IPython.display import Audio, display

from torchsig.utils.dsp import compute_spectrogram
from torchsig.utils.file_handlers import WAVReader

## Download and play two short speech recordings

Both clips contain recognizable English speech. Re-running this cell reuses files already on disk.

In [ ]:
download_root = Path("data/downloaded_speech")
dataset_root = Path("data/audio_record_manifest_demo")
nested_dir = dataset_root / "nested"
download_root.mkdir(parents=True, exist_ok=True)
nested_dir.mkdir(parents=True, exist_ok=True)

downloads = {
    download_root / "tutorial_speech.wav": "https://download.pytorch.org/torchaudio/tutorial-assets/Lab41-SRI-VOiCES-src-sp0307-ch127535-sg0042.wav",
    download_root / "jfk.flac": "https://raw.githubusercontent.com/openai/whisper/main/tests/jfk.flac",
}

for destination, url in downloads.items():
    if not destination.exists():
        urlretrieve(url, destination)
    info = sf.info(destination)
    print(f"{destination}: {info.frames} frames, {info.channels} channels, {info.samplerate} Hz")

In [ ]:
for path in downloads:
    samples, sample_rate = sf.read(path, dtype="float32", always_2d=True)
    print(path.name)
    display(Audio(samples.T, rate=sample_rate))

## Build the MR2 record manifest

Each metadata row now identifies its audio file and exact frame interval. The first file contains two records of different sizes, while the second contains one. Gaps are allowed; overlapping or out-of-bounds intervals are rejected when the reader is created. Paths are relative to the dataset root, including the nested path.

In [ ]:
iq_paths = [nested_dir / "tutorial_speech_iq.wav", dataset_root / "jfk_iq.wav"]
for source_path, iq_path in zip(downloads, iq_paths, strict=True):
    source, sample_rate = sf.read(source_path, dtype="float32", always_2d=True)
    mono = source[:, 0]
    analytic = hilbert(mono).astype(np.complex64)
    iq_stereo = np.column_stack((analytic.real, analytic.imag))
    sf.write(iq_path, iq_stereo, sample_rate, subtype="FLOAT")

tutorial_info = sf.info(iq_paths[0])
jfk_info = sf.info(iq_paths[1])
split = tutorial_info.frames // 3

records = [
    (0, "tutorial_opening", "nested/tutorial_speech_iq.wav", 0, split, tutorial_info.samplerate),
    (1, "tutorial_remainder", "nested/tutorial_speech_iq.wav", split, tutorial_info.frames - split, tutorial_info.samplerate),
    (2, "jfk_speech", "jfk_iq.wav", 0, jfk_info.frames, jfk_info.samplerate),
]

fieldnames = [
    "index", "label", "modcod", "sample_rate",
    "file_path", "start_frame", "num_frames",
    "expected_sample_rate", "channel_count",
]
with (dataset_root / "metadata.csv").open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()
    for index, label, file_path, start_frame, num_frames, sample_rate in records:
        writer.writerow({
            "index": index,
            "label": label,
            "modcod": 0,
            "sample_rate": sample_rate,
            "file_path": file_path,
            "start_frame": start_frame,
            "num_frames": num_frames,
            "expected_sample_rate": sample_rate,
            "channel_count": 2,
        })

records

## Read the variable-length records with TorchSIG

`WAVReader` validates every descriptor while constructing its shared audio-record index. Each `read()` result is a TorchSIG `Signal` with complex64 IQ data and its corresponding metadata row.

In [ ]:
reader = WAVReader(dataset_root)
signals = [reader.read(index) for index in range(len(reader))]

for index, signal in enumerate(signals):
    descriptor = reader.record_layout[index]
    print(
        f"record {index}: {descriptor.path.relative_to(dataset_root.resolve())}, "
        f"frames [{descriptor.start_frame}:{descriptor.start_frame + descriptor.num_frames}], "
        f"shape={signal.data.shape}, dtype={signal.data.dtype}"
    )

## Display the TorchSIG signals

The plots show each variable-length record as complex IQ data: I/Q over time, its constellation, and a TorchSIG-computed spectrogram.

In [ ]:
fig, axes = plt.subplots(len(signals), 3, figsize=(14, 3 * len(signals)))

for index, (signal, row_axes) in enumerate(zip(signals, axes, strict=True)):
    iq = signal.data
    sample_rate = float(signal.metadata["sample_rate"])
    time_ms = np.arange(iq.size) / sample_rate * 1_000

    row_axes[0].plot(time_ms, iq.real, label="I", linewidth=1)
    row_axes[0].plot(time_ms, iq.imag, label="Q", linewidth=1, alpha=0.8)
    row_axes[0].set(title=f"Record {index}: {signal.metadata['label']}", xlabel="Time (ms)")
    row_axes[0].legend()

    row_axes[1].scatter(iq.real, iq.imag, s=8, alpha=0.6)
    row_axes[1].set(xlabel="I", ylabel="Q", title="IQ plane")
    row_axes[1].axis("equal")

    fft_size = min(64, 2 ** int(np.floor(np.log2(iq.size))))
    spectrum = compute_spectrogram(iq, fft_size=fft_size, fft_stride=max(1, fft_size // 4))
    row_axes[2].imshow(spectrum, origin="lower", aspect="auto", cmap="viridis")
    row_axes[2].set(title="TorchSIG spectrogram", xlabel="Frame", ylabel="Frequency bin")

fig.tight_layout()
plt.show()

The records above are indexed by metadata rather than by `file_index + element_offset * num_iq_samples`. Changing a row's non-overlapping `start_frame` or `num_frames` is enough to select a different segment; no uniform record size or uniform records-per-file setting is required.